In [1]:
import os
import shutil

# 1. Clone repository if running in a fresh Colab session
if not os.path.exists('SeasonAQ'):
    !git clone https://github.com/nashatislam/SeasonAQ.git

# Change working directory into the repository root
%cd /content/SeasonAQ

# 2. Define exact model folder names and dataset filenames
data_files = ['data_winter.csv', 'data_summer.csv', 'data_monsoon.csv', 'data_post_monsoon.csv']
model_dirs = ['logistic_regression', 'Decision_Tree', 'random_forest', 'KNN', 'naive_bayes']

# 3. Correct source path pointing to repository dataset folder
source_dir = '/content/SeasonAQ/datasets/datasets_Nashat'

# 4. Copy datasets to repository root AND each model subdirectory
for file_name in data_files:
    src_path = os.path.join(source_dir, file_name)
    if os.path.exists(src_path):
        # Copy to repository root
        shutil.copy(src_path, os.path.join('/content/SeasonAQ', file_name))

        # Copy to each individual model directory
        for model in model_dirs:
            target_dir = os.path.join('/content/SeasonAQ/models', model)
            if os.path.exists(target_dir):
                shutil.copy(src_path, os.path.join(target_dir, file_name))

# 5. Prepare output directories
os.makedirs("results/tables/e2", exist_ok=True)
os.makedirs("results/visualizations/e2", exist_ok=True)
os.makedirs("results/visualizations/cross_eval_model_wise", exist_ok=True)

print("✅ Step 1 Complete: Datasets copied from 'datasets/datasets_Nashat/' to all model subdirectories.")

[WinError 3] The system cannot find the path specified: '/content/SeasonAQ'
c:\Users\ACER\Desktop\SeasonAQ\models
✅ Step 1 Complete: Datasets copied from 'datasets/datasets_Nashat/' to all model subdirectories.


Cloning into 'SeasonAQ'...


In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from IPython.utils.capture import capture_output

model_config = [
    ('logistic_regression', 'Logistic Regression'),
    ('Decision_Tree', 'Decision Tree'),
    ('random_forest', 'Random Forest'),
    ('KNN', 'k-NN'),
    ('naive_bayes', 'Naive Bayes'),
]

seasons = ['winter', 'summer', 'monsoon', 'post_monsoon']
all_summary_matrices = {}
eval_records = []

root_dir = '/content/SeasonAQ'

print("--- Starting Silent Cross-Season Evaluation Loop (E2) ---")

for model_folder, display_name in model_config:
    model_path = os.path.join(root_dir, "models", model_folder)
    notebook_file = "01_train_X_test_X.ipynb"
    notebook_full_path = os.path.join(model_path, notebook_file)

    if not os.path.exists(notebook_full_path):
        print(f"[SKIP] Could not find notebook: {notebook_full_path}")
        continue

    print(f"Processing [{display_name}]...")

    try:
        # Switch working directory to model folder so %run resolves relative paths locally
        os.chdir(model_path)

        # 1. Silently run the training notebook
        with capture_output():
            %run 01_train_X_test_X.ipynb

        # 2. Compute 4x4 Transferability Matrix
        matrix_f1 = pd.DataFrame(index=seasons, columns=seasons, dtype=float)

        for train_s in seasons:
            clf = trained_models[train_s]

            for test_s in seasons:
                X_test = processed_data[test_s]['X_test']
                y_test = processed_data[test_s]['y_test']

                y_pred = clf.predict(X_test)
                score = round(f1_score(y_test, y_pred, average='macro'), 4)
                matrix_f1.loc[train_s, test_s] = score

                eval_records.append({
                    'Algorithm': display_name,
                    'Train Season': train_s.capitalize().replace('_m', '_M'),
                    'Test Season': test_s.capitalize().replace('_m', '_M'),
                    'Macro F1': float(score)
                })

        all_summary_matrices[display_name] = matrix_f1

        # 3. Save matrix table to CSV
        csv_path = os.path.join(root_dir, f"results/tables/e2/matrix_{model_folder.lower()}.csv")
        matrix_f1.to_csv(csv_path)
        print(f"  --> Saved Matrix CSV for {display_name}.")

    finally:
        # Restore root working directory
        os.chdir(root_dir)

# Save consolidated evaluation dataframe
df_eval = pd.DataFrame(eval_records)
df_eval.to_csv("results/tables/e2/cross_eval_longform.csv", index=False)

print("\n✅ Step 2 Complete: All 80 cross-evaluations executed successfully.")

ModuleNotFoundError: No module named 'numpy'

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np # Import numpy as it is used for np.isnan

sns.set_theme(style="whitegrid")

train_seasons_cap = ['Winter', 'Summer', 'Monsoon', 'Post_monsoon']
palette = ['#C77CFF', '#FEA903', '#C7E171', '#719EFF']

print("--- Generating Model-Wise Cross-Evaluation Bar Charts ---")

for m_name in df_eval['Algorithm'].unique():
    df_plot_model = df_eval[df_eval['Algorithm'] == m_name]

    fig, ax = plt.subplots(figsize=(10, 6))

    sns.barplot(
        data=df_plot_model,
        x='Test Season',
        y='Macro F1',
        hue='Train Season',
        hue_order=train_seasons_cap,
        palette=palette,
        ax=ax,
        edgecolor='gray',
        linewidth=0.5
    )

    ax.set_title(f'{m_name}: Cross-Seasonal Evaluation', fontweight='bold', fontsize=14, pad=12)
    ax.set_xlabel('Test Season', fontweight='bold', fontsize=11)
    ax.set_ylabel('Macro F1 Score', fontweight='bold', fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', labelsize=10)
    ax.tick_params(axis='y', labelsize=10)

    # Ordering references for x-axis categories and hue groups
    test_season_order = df_plot_model['Test Season'].unique()
    train_season_hue_order = train_seasons_cap

    # Iterate through each hue group container and annotate / cross-hatch
    for hue_idx, container in enumerate(ax.containers):
        current_train_season = train_season_hue_order[hue_idx]
        for bar_idx, p in enumerate(container.patches):
            height = p.get_height()
            current_test_season = test_season_order[bar_idx]

            # Annotate with F1 score
            if not np.isnan(height) and height > 0:
                ax.annotate(
                    f'{height:.2f}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    fontsize=7.5, color='black',
                    xytext=(0, 2),
                    textcoords='offset points'
                )

            # Cross-hatch within-season baseline bars (Train Season == Test Season)
            if current_test_season == current_train_season:
                p.set_hatch('///')

    ax.legend(title='Train Season', frameon=True, loc='upper right')
    plt.tight_layout()

    # Save visualization PNG
    clean_m_name = m_name.lower().replace(' ', '_').replace('-', '_')
    plot_path_model = f"results/visualizations/cross_eval_model_wise/{clean_m_name}_cross_eval.png"
    plt.savefig(plot_path_model, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {plot_path_model}")

print("\n✅ Step 3 Complete: Cross-seasonal evaluation visualizations generated for each model.\n")

In [ ]:
from google.colab import files

print("--- Triggering Browser Downloads for E2 Visualization Artifacts ---")

download_files = [
    "results/tables/e2/cross_eval_longform.csv"
]

# Append individual model plots
for m_name in df_eval['Algorithm'].unique():
    clean_m_name = m_name.lower().replace(' ', '_').replace('-', '_')
    download_files.append(f"results/visualizations/cross_eval_model_wise/{clean_m_name}_cross_eval.png")

for file_path in download_files:
    if os.path.exists(file_path):
        files.download(file_path)

print("✅ Execution Complete. All tables and plots sent to download queue.")